<a href="https://colab.research.google.com/github/halilcankaroglu/yzm511-midterm-mlp/blob/main/03_final_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os, json
os.chdir("/content")
if not os.path.exists("yzm511-midterm-mlp"):
    !git clone https://github.com/halilcankaroglu/yzm511-midterm-mlp.git
%cd /content/yzm511-midterm-mlp

from google.colab import userdata
token = userdata.get("GH_TOKEN")
!git remote set-url origin https://{token}@github.com/halilcankaroglu/yzm511-midterm-mlp.git
!git config user.name "Halil Can Karoğlu"
!git config user.email "GITHUB_EPOSTAN"

SEEDS = json.load(open("configs/seeds.json"))

Cloning into 'yzm511-midterm-mlp'...
remote: Enumerating objects: 71, done.
remote: Counting objects: 100% (71/71), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 71 (delta 15), reused 48 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (71/71), 967.47 KiB | 6.81 MiB/s, done.
Resolving deltas: 100% (15/15), done.
/content/yzm511-midterm-mlp


Veri

In [ ]:
import numpy as np, pandas as pd

df = pd.read_csv("data/bank-additional.csv", sep=";").drop(columns=["duration"])
split = pd.read_csv("results/dataset_split.csv")
dev_rows = split.loc[split["set"] == "dev", "row_index"].values
test_rows = split.loc[split["set"] == "test", "row_index"].values

X_dev = df.iloc[dev_rows].drop(columns=["y"]).reset_index(drop=True)
y_dev = (df.iloc[dev_rows]["y"] == "yes").astype(int).values
X_test = df.iloc[test_rows].drop(columns=["y"]).reset_index(drop=True)   # etiket yok
print(X_dev.shape, X_test.shape)

(3295, 19) (824, 19)


Tam development üzerinde iç CV (36 eğitim)

In [ ]:
from src.nested_cv import run_search

manual = json.load(open("configs/manual_candidates.json"))
rand = json.load(open("configs/random_candidates.json"))
training = json.load(open("configs/training.json"))

sel, rows_all = {}, []
for method, cl in [("manual", manual), ("random", rand)]:
    rows, _, best = run_search(cl, X_dev, y_dev, training, SEEDS,
                               dict(outer="full_dev", method=method))
    rows_all.append(rows)
    sel[method] = (best, float(rows["mean_ap"].max()))
    print(f"{method}: {best} | iç CV mean AP {sel[method][1]:.4f}")
pd.concat(rows_all).to_csv("results/final_selection_results.csv", index=False)

method = max(sel, key=lambda m: sel[m][1])          # önceden tanımlı kural
code = sel[method][0]
cfg = {c["code"]: c for c in manual + rand}[code]
print("\nNihai:", method, code, cfg)

manual: M2 | iç CV mean AP 0.4250
random: R5 | iç CV mean AP 0.4230

Nihai: manual M2 {'optimizer': 'adam', 'hidden_layers': [32], 'activation': 'relu', 'scaler': 'standard', 'lr': 0.001, 'weight_decay': 0.0001, 'batch_size': 64, 'dropout': 0.0, 'clipping': False, 'scheduler': True, 'code': 'M2'}


Üç seed ile eğitim ve kilitli tahminler

In [ ]:
import torch, joblib
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from src.mlp import fit_mlp, predict_proba
from src.preprocessing import build_preprocessor

os.makedirs("models", exist_ok=True)
preds = pd.DataFrame({"row_index": test_rows})
runs = []
for i, s in enumerate(SEEDS["init"]):
    res = fit_mlp(cfg, X_dev, y_dev, training, SEEDS["early_stop"], s)
    torch.save(res["model"].state_dict(), f"models/mlp_seed{i}.pt")
    joblib.dump(res["pre"], f"models/mlp_seed{i}_pre.joblib")
    if i == 0:
        res["history"].to_csv("results/final_training_history.csv", index=False)
    p = predict_proba(res, X_test)
    preds[f"p_mlp_seed{i}"] = p
    preds[f"pred_mlp_seed{i}"] = (p >= training["threshold"]).astype(int)
    runs.append(dict(seed=s, best_epoch=res["best_epoch"], epochs_run=res["epochs_run"],
                     seconds=round(res["seconds"], 1), n_params=res["n_params"]))

logreg = Pipeline([("pre", build_preprocessor("standard")),
                   ("clf", LogisticRegression(max_iter=2000))]).fit(X_dev, y_dev)
joblib.dump(logreg, "models/logreg.joblib")
p = logreg.predict_proba(X_test)[:, 1]
preds["p_logreg"], preds["pred_logreg"] = p, (p >= training["threshold"]).astype(int)

assert "y" not in preds.columns
preds.to_csv("results/test_predictions_locked.csv", index=False)

final = dict(selected_method=method, selected_code=code, config=cfg,
             inner_cv_mean_ap=sel[method][1],
             search_winners={m: dict(code=c, mean_ap=a) for m, (c, a) in sel.items()},
             primary_seed=SEEDS["init"][0], seeds=SEEDS, threshold=training["threshold"],
             training=training, seed_runs=runs)
json.dump(final, open("results/final_hyperparameters.json", "w"), indent=2, ensure_ascii=False)
pd.DataFrame(runs)

,seed,best_epoch,epochs_run,seconds,n_params
0,290162036,14,29,4.7,2017
1,290162037,12,27,2.0,2017
2,290162038,14,29,2.0,2017


In [ ]:
!git pull --rebase
!git add results/final_selection_results.csv results/final_hyperparameters.json \
         results/final_training_history.csv results/test_predictions_locked.csv models/
!git commit -m "KİLİT: nihai konfigürasyon ve etiketsiz test tahminleri"
!git push

error: cannot pull with rebase: You have unstaged changes.
error: Please commit or stash them.
[main 36cdbe6] KİLİT: nihai konfigürasyon ve etiketsiz test tahminleri
 11 files changed, 950 insertions(+)
 create mode 100644 models/logreg.joblib
 create mode 100644 models/mlp_seed0.pt
 create mode 100644 models/mlp_seed0_pre.joblib
 create mode 100644 models/mlp_seed1.pt
 create mode 100644 models/mlp_seed1_pre.joblib
 create mode 100644 models/mlp_seed2.pt
 create mode 100644 models/mlp_seed2_pre.joblib
 create mode 100644 results/final_hyperparameters.json
 create mode 100644 results/final_selection_results.csv
 create mode 100644 results/final_training_history.csv
 create mode 100644 results/test_predictions_locked.csv
Enumerating objects: 15, done.
Counting objects: 100% (15/15), done.
Delta compression using up to 2 threads
Compressing objects: 100% (13/13), done.
Writing objects: 100% (13/13), 54.28 KiB | 1.94 MiB/s, done.
Total 13 (delta 3), reused 0 (delta 0), pack-reused 0
remot

In [ ]:
!printf "__pycache__/\n*.pyc\n.ipynb_checkpoints/\n" > .gitignore
!git pull --rebase
!git rm -r --cached -q src/__pycache__
!git add .gitignore
!git commit -m ".gitignore eklendi, __pycache__ kaldırıldı"
!git push

error: cannot pull with rebase: You have unstaged changes.
error: Please commit or stash them.
[main d143a34] .gitignore eklendi, __pycache__ kaldırıldı
 3 files changed, 3 insertions(+)
 create mode 100644 .gitignore
 delete mode 100644 src/__pycache__/mlp.cpython-313.pyc
 delete mode 100644 src/__pycache__/preprocessing.cpython-313.pyc
Enumerating objects: 6, done.
Counting objects: 100% (6/6), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (4/4), 411 bytes | 411.00 KiB/s, done.
Total 4 (delta 2), reused 0 (delta 0), pack-reused 0
To https://github.com/halilcankaroglu/yzm511-midterm-mlp.git
   36cdbe6..d143a34  main -> main
